# Session 28: Sampling and Confidence Intervals

Data Science with Python, Level 2, Stage 4: Statistics

## Learning Objectives

By the end of this session you will be able to:

1. Explain why sample results vary.
2. Describe the sampling distribution of the mean.
3. Calculate a standard error.
4. Build a 95 percent confidence interval for a mean.
5. Build one for a proportion.
6. Use the bootstrap when formulas do not fit.

## Where We Are

In Session 27 you saw that the means of many samples of dice rolls formed a bell curve, even though a single die is flat. That is the key to most of statistics.

We almost never see the whole population. A survey asks 500 people, not a whole country; a test runs for two weeks, not forever. Every sample gives a slightly different answer. A **confidence interval** turns one sample's answer into a range that very probably contains the true population value, and says how sure we are.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(28)
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
exams = pd.read_csv(DATA_URL + "exam_scores.csv")

## 1. Samples Vary

To see sampling in action we invent a **population** we fully know: the commute times, in minutes, of 50,000 workers in a city. They are right-skewed, like many real measurements. Then we take samples and compare their means with the true mean.

In [ ]:
population = rng.gamma(shape=2.0, scale=15.0, size=50_000)
print("True mean:", round(population.mean(), 2))

for i in range(5):
    sample = rng.choice(population, size=40, replace=False)
    print(f"Sample {i + 1} mean: {sample.mean():.2f}")

## 2. The Sampling Distribution

In [ ]:
means_40 = [rng.choice(population, 40).mean() for _ in range(5000)]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.histplot(population, bins=50, ax=axes[0])
axes[0].set_title("Population: skewed")
sns.histplot(means_40, bins=50, ax=axes[1], color="tab:purple")
axes[1].set_title("Means of 5,000 samples of 40: a bell")
plt.show()

In [ ]:
for n in [10, 40, 160]:
    means = [rng.choice(population, n).mean() for _ in range(3000)]
    print(f"n = {n:>3}: spread of means {np.std(means):.2f}, "
          f"formula {population.std() / np.sqrt(n):.2f}")

## 3. Confidence Intervals for a Mean

Because sample means are bell-shaped, about 95 percent of them fall within about 1.96 standard errors of the true mean. Turning this around: if we take the sample mean and go 1.96 standard errors either side, the interval catches the true mean in about 95 percent of samples. That interval is a **95 percent confidence interval**.

For small samples we use a slightly wider multiplier from the **t distribution** instead of 1.96. `stats.t.interval` does it for us.

In [ ]:
math = exams["math"]
n, mean = len(math), math.mean()
se = math.std() / np.sqrt(n)

print(f"Mean {mean:.2f}, standard error {se:.2f}")
print(f"Approximate 95% CI: {mean - 1.96 * se:.2f} to {mean + 1.96 * se:.2f}")
low, high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
print(f"t-based 95% CI:     {low:.2f} to {high:.2f}")

In [ ]:
hits = 0
for _ in range(1000):
    s = rng.choice(population, 40)
    se = s.std(ddof=1) / np.sqrt(40)
    low, high = stats.t.interval(0.95, df=39, loc=s.mean(), scale=se)
    hits += low <= population.mean() <= high

print("Intervals that caught the true mean:", hits / 1000)

## 4. Confidence Intervals for a Proportion

In [ ]:
p_hat = exams["passed"].mean()
n = len(exams)
se = np.sqrt(p_hat * (1 - p_hat) / n)

print(f"Pass rate {p_hat:.3f}, margin of error {1.96 * se:.3f}")
print(f"95% CI: {p_hat - 1.96 * se:.3f} to {p_hat + 1.96 * se:.3f}")

In [ ]:
for n in [100, 400, 1600]:
    margin = 1.96 * np.sqrt(0.5 * 0.5 / n)
    print(f"n = {n:>4}: margin of error about +-{margin:.1%}")

## 5. The Bootstrap

Formulas exist for means and proportions, but not for every statistic, such as the median. The **bootstrap** imitates taking new samples by **resampling** the sample you have, with replacement, thousands of times. The middle 95 percent of the resampled statistics is a confidence interval.

In [ ]:
flats = pd.read_csv(DATA_URL + "apartments.csv")
prices = flats["price_k"].to_numpy()

boot = [np.median(rng.choice(prices, size=len(prices))) for _ in range(5000)]
low, high = np.percentile(boot, [2.5, 97.5])
print(f"Median price {np.median(prices):.1f}, 95% CI {low:.1f} to {high:.1f}")

sns.histplot(boot, bins=40)
plt.title("Bootstrap medians of apartment prices")
plt.show()

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Wrong claim | Reading one interval as a 95 percent chance | Say: 95 percent of such intervals contain the true value. |
| Too precise | Reporting a sample mean with no interval | Give the interval, or at least the margin of error. |
| Biased sample | Using a large but non-random sample | No formula fixes bias; only random sampling does. |
| Wrong spread | Using the std instead of the standard error | The interval uses std divided by the square root of n. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: English Interval

Build a 95 percent confidence interval for the mean English score with `stats.t.interval`.

In [ ]:
# Write your code below

### In-Class Exercise 2: Small Sample

Take a random sample of 25 students and build the same interval for the math score. How much wider is it than the interval from all 500?

In [ ]:
# Write your code below

### In-Class Exercise 3: Churn Rate

Load `telecom_churn.csv` and build a 95 percent confidence interval for the churn rate of customers on monthly contracts.

In [ ]:
# Write your code below

### In-Class Exercise 4: Bootstrap a Mean

Use the bootstrap to build a 95 percent interval for the mean math score, and compare it with the t interval from the lecture.

In [ ]:
# Write your code below

## Summary

- Every sample gives a slightly different estimate.
- Sample means are bell-shaped: the central limit theorem.
- Standard error = std / sqrt(n); it shrinks with bigger samples.
- 95 percent CI: estimate +- about 2 standard errors.
- For proportions, SE = sqrt(p (1 - p) / n).
- The bootstrap gives intervals for any statistic.

## What's Next

So far we described one column at a time. Next session measures how two columns move together, and shows why a strong correlation still does not prove that one causes the other.